In [2]:
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# auto-load FAISS if exists
try:
    index = faiss.read_index("jobs.index")
    with open("jobs_data.pkl", "rb") as f:
        df = pickle.load(f)

    print("✅ FAISS + dataset loaded")

except:
    print("⚠️ FAISS not found — run build_faiss() once")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚠️ FAISS not found — run build_faiss() once


In [3]:
# Step 1: Install dependencies
!pip install sentence-transformers pandas numpy tqdm

import pandas as pd
import numpy as np
import json
from tqdm import tqdm

# ── Sample job data (replace this with your real 97k dataset) ──────────────
jobs_raw = [
    {
        "id": 1,
        "title": "Senior Data Scientist",
        "company": "Google",
        "description": "We are looking for a Senior Data Scientist with experience in Python, machine learning, and SQL. You will build predictive models, work with large datasets using Spark, and deploy models to production using Docker and Kubernetes."
    },
    {
        "id": 2,
        "title": "Frontend Engineer",
        "company": "Airbnb",
        "description": "Join our team as a Frontend Engineer. You should have strong skills in React, TypeScript, and CSS. Experience with GraphQL, REST APIs, and testing frameworks like Jest is a plus."
    },
    {
        "id": 3,
        "title": "DevOps Engineer",
        "company": "Netflix",
        "description": "We need a DevOps Engineer skilled in AWS, Terraform, CI/CD pipelines, Docker, and Kubernetes. You will automate infrastructure, monitor systems using Prometheus and Grafana, and ensure 99.9% uptime."
    },
    {
        "id": 4,
        "title": "Machine Learning Engineer",
        "company": "Meta",
        "description": "Looking for an ML Engineer with expertise in PyTorch, model training at scale, feature engineering, and MLflow for experiment tracking. Python and familiarity with distributed training required."
    },
    {
        "id": 5,
        "title": "Backend Engineer",
        "company": "Stripe",
        "description": "Backend Engineer role requiring strong knowledge of Python or Go, PostgreSQL, Redis, REST API design, and microservices architecture. Experience with Kafka and event-driven systems is a plus."
    },
]

# ── Load into DataFrame ────────────────────────────────────────────────────
df = pd.DataFrame(jobs_raw)
print(f"✅ Loaded {len(df)} jobs\n")
print(df[["id", "title", "company"]])

✅ Loaded 5 jobs

   id                      title  company
0   1      Senior Data Scientist   Google
1   2          Frontend Engineer   Airbnb
2   3            DevOps Engineer  Netflix
3   4  Machine Learning Engineer     Meta
4   5           Backend Engineer   Stripe


In [2]:
# Install additional dependency for local LLM inference
!pip install transformers sentencepiece --quiet

In [4]:
import pandas as pd
import json
import re
from tqdm import tqdm
from transformers import pipeline

print('✅ Imports successful')

✅ Imports successful


In [5]:
# ── Test on one job first ──────────────────────────────────────────────────
sample_desc = df.loc[0, 'description']
prompt = build_prompt(sample_desc)

result = extractor(prompt)[0]['generated_text']
skills = parse_skills(result)

print(f'Job   : {df.loc[0, "title"]} @ {df.loc[0, "company"]}')
print(f'Raw   : {result}')
print(f'Skills: {skills}')

NameError: name 'build_prompt' is not defined

In [23]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [6]:
import pandas as pd

# Load Excel dataset
df = pd.read_excel(
    "C:/Users/hp/Downloads/resume_extraction_2/indian-job-market-dataset-2025.xlsx"
)

# Show basic information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Display first 5 rows
df.head()

Shape: (97929, 17)

Columns:
['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience']


,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Pow...",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000.0,400000.0,2.0,4.0
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162.0,4.0,"Ensure active Fire Protection System,such as F...",300000.0,500000.0,6.0,11.0
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth ...",12-18 Yrs,Not disclosed,Chennai,1324750,2892.0,4.2,MBA Marketing (preferred Tier II or III B- Sch...,0.0,0.0,12.0,18.0
3,Medical Billing Executive,270925007443,INR,6 Days Ago,GNR Global Services,"Fluent English,Spoken English,Good English Com...",0-3 Yrs,"70,000-2 Lacs PA","Mohali, Chandigarh, Kharar, Zirakpur",123804403,NaN,NaN,Job Title-Medical Billing Executive\nLocation-...,70000.0,200000.0,0.0,3.0
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephr...",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134.0,3.4,Principal Tasks & Responsibilities : (Please w...,800000.0,1800000.0,5.0,10.0


In [7]:
df = df.fillna("")

df["job_text"] = (
    "TITLE: " + df["title"].astype(str) + "\n" +
    "COMPANY: " + df["companyName"].astype(str) + "\n" +
    "SKILLS: " + df["tagsAndSkills"].astype(str) + "\n" +
    "EXPERIENCE: " + df["experience"].astype(str) + "\n" +
    "LOCATION: " + df["location"].astype(str) + "\n" +
    "DESCRIPTION: " + df["jobDescription"].astype(str)

)

In [8]:
import re
import pandas as pd

# Fill missing values
df = df.fillna("")

# Clean each field separately
def clean_text(text):
    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Normalize bullet symbols
    text = re.sub(r"[•●▪◆►■]", " ", text)

    # Add spaces after commas in skills
    text = text.replace(",", ", ")

    # Remove multiple spaces but KEEP newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse 3+ newlines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# Apply cleaning to each field
df["title"] = df["title"].apply(clean_text)
df["companyName"] = df["companyName"].apply(clean_text)
df["tagsAndSkills"] = df["tagsAndSkills"].apply(clean_text)
df["experience"] = df["experience"].apply(clean_text)
df["location"] = df["location"].apply(clean_text)
df["jobDescription"] = df["jobDescription"].apply(clean_text)

# Create structured job text
df["job_text"] = (
    "TITLE:\n" + df["title"] + "\n\n"
    "COMPANY:\n" + df["companyName"] + "\n\n"
    "SKILLS:\n" + df["tagsAndSkills"] + "\n\n"
    "EXPERIENCE:\n" + df["experience"] + "\n\n"
    "LOCATION:\n" + df["location"] + "\n\n"
    "DESCRIPTION:\n" + df["jobDescription"]
)

# Preview
print("Dataset shape:", df.shape)
print("\nSample job_text:\n")
print(df["job_text"].iloc[0])

Dataset shape: (97929, 18)

Sample job_text:

TITLE:
Sr. HR Recruiter (NON IT)

COMPANY:
Orion

SKILLS:
Communication, Manpower, Staffing, Convincing Power, Hiring, Recruitment, SR, Communication skills

EXPERIENCE:
2-4 Yrs

LOCATION:
Kolkata(Chinar Park)

DESCRIPTION:
Preferred candidate profile . .


In [9]:
# Core libraries for FAISS resume-job matching system
!pip install sentence-transformers
!pip install faiss-cpu
!pip install numpy
!pip install pandas

# OPTIONAL (for next steps - resume PDF parsing)
!pip install pdfplumber



In [46]:
import faiss
import numpy as np
import pickle

job_texts = df["job_text"].tolist()

job_embeddings = model.encode(
    job_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    batch_size=128,
    show_progress_bar=True
).astype("float32")

dimension = job_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(job_embeddings)

faiss.write_index(index, "jobs.index")

# SAVE metadata
with open("jobs_meta.pkl", "wb") as f:
    pickle.dump(df.to_dict("records"), f)

# SAVE embeddings (VERY IMPORTANT FOR SPEED)
np.save("job_embeddings.npy", job_embeddings)

print("Saved EVERYTHING (index + metadata + embeddings)")

Batches:   0%|          | 0/766 [00:00<?, ?it/s]

Saved EVERYTHING (index + metadata + embeddings)


In [10]:
import faiss
import pickle
import numpy as np

# load FAISS index (instant)
index = faiss.read_index("jobs.index")

# load metadata (instant)
with open("jobs_meta.pkl", "rb") as f:
    job_data = pickle.load(f)

# load embeddings (NO recompute needed)
job_embeddings = np.load("job_embeddings.npy")

df = job_data  # replace your df if needed

print("Loaded everything instantly")

Loaded everything instantly


In [11]:
def search_jobs(resume_text, top_k=20):

    query = "query: " + resume_text

    resume_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32").reshape(1, -1)

    scores, indices = index.search(resume_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        job = df[idx]  # IMPORTANT: now list, not df.iloc

        results.append({
            "score": float(score),
            "title": job["title"],
            "company": job["companyName"],
            "location": job["location"],
            "experience": job["experience"],
            "skills": job["tagsAndSkills"],
            "job_text": job["job_text"]
        })

    return results

In [12]:
resume_path = "C:/Users/hp/Downloads/ANSHULA CHAUHAN CV.pdf"

In [13]:
import fitz  # PyMuPDF
import docx

def extract_resume_text(file_path):
    """
    Extract text from PDF or DOCX resume.
    """

    if file_path.lower().endswith(".pdf"):

        doc = fitz.open(file_path)

        text = ""

        for page in doc:
            text += page.get_text()

        doc.close()

    elif file_path.lower().endswith(".docx"):

        doc = docx.Document(file_path)

        text = "\n".join([para.text for para in doc.paragraphs])

    else:
        raise ValueError("Only PDF and DOCX files are supported.")

    return text

In [14]:
import re

def clean_resume(text):

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"http\S+|www\S+", " ", text)

    text = re.sub(r"[•●▪◆►■]", " ", text)

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()
resume_text = extract_resume_text(resume_path)

resume_text = clean_resume(resume_text)

print("========== EXTRACTED RESUME ==========\n")

print(resume_text[:3000])   # Preview first 3000 characters

========== EXTRACTED RESUME ==========

ANSHULA CHAUHAN 
Yamunanagar, Haryana 135001 
anshulachauhan17@gmail.com 
+9199960 93850 
 
A dedicated and passionate PhD scholar in Environmental 
Science with a solid academic foundation and research 
background focused on sustainable environmental 
management, as well as ecosystem conservation. 
Committed to driving an impactful change, my goal is to 
bridge the gap between scientific research and practical 
environmental solutions that promote long-term 
ecological balance. 
 
 
Personal Details 
 
Date of Birth: 1999-08-17 
 
 
Work Experience 
 
Environmental Consultant 
Eco Paryavaran Laboratories and Consultants Private Limited, Mohali, Punjab- 160055. 
February 2024 to June 2024 
Worked as an ‘Environmental Scientist cum Consultant’ at Eco Paryavaran Laboratories and Consultants 
Private Limited, Mohali in the EMS division in sectors dealing with Six monthly Compliance reports, EIA 
Reports, Groundwater extraction NOCs. 
Internship 
Mun

In [15]:
# ==========================================
# STEP 5: Generate Resume Embedding
# ==========================================

resume_embedding = model.encode(
    resume_text,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

# Reshape for FAISS
resume_embedding = resume_embedding.reshape(1, -1)

print("Resume embedding shape:", resume_embedding.shape)

Resume embedding shape: (1, 384)


In [17]:
import numpy as np
def retrieve_jobs(resume_embedding, top_k=30):

    resume_embedding = np.array(resume_embedding).astype("float32").reshape(1, -1)

    scores, indices = index.search(resume_embedding, top_k)

    retrieved_jobs = []

    for score, idx in zip(scores[0], indices[0]):

        job = df[int(idx)]

        retrieved_jobs.append({
            "similarity_score": round(float(score), 4),
            "title": job["title"],
            "company": job["companyName"],
            "location": job["location"],
            "experience": job["experience"],
            "skills": job["tagsAndSkills"],
            "description": job["jobDescription"]
        })

    return retrieved_jobs
top_jobs = retrieve_jobs(resume_embedding, top_k=30)
for i, job in enumerate(top_jobs[:5], 1):
    print("=" * 80)
    print(f"Rank #{i}")
    print("Similarity :", job["similarity_score"])
    print("Title      :", job["title"])
    print("Company    :", job["company"])
    print("Location   :", job["location"])
    print("Experience :", job["experience"])
    print("Skills     :", job["skills"])

Rank #1
Similarity : 0.6117
Title      : Senior Project Consultant
Company    : EY
Location   : Mumbai
Experience : 7-12 Yrs
Skills     : Assurance, Project finance, Financial planning, Corporate finance, Investment banking, Operations, Advisory, Analytics
Rank #2
Similarity : 0.5991
Title      : Strategic Development Manager Green Hydrogen and Ammonia
Company    : Aelius Consultants Delhi
Location   : Noida
Experience : 5-10 Yrs
Skills     : Business & Commercial Strategy, Strategic Procurement, Offtake Development, Market Analysis, Market Intelligence, Commercial, Strategy development, Hydrogen
Rank #3
Similarity : 0.5724
Title      : Hiring_ Chemical Engineer_ Male Fresher_Water Treatment_ Delhi & Vizag
Company    : Seven Consultancy
Location   : Visakhapatnam, New Delhi
Experience : 0-1 Yrs
Skills     : Filtration, Reverse Osmosis, Water Treatment, tds, WTP, Chemical Testing, RO System, Zero Liquid Discharge
Rank #4
Similarity : 0.5723
Title      : Engineer/Associate counsellor-Wat

In [18]:
def create_prompt(resume_text, top_jobs):

    prompt = f"""
You are an expert AI recruiter.

Analyze the candidate's resume and compare it with the retrieved jobs.

Resume:
{resume_text}

Retrieved Jobs:
"""

    for i, job in enumerate(top_jobs, 1):

        prompt += f"""

Job {i}

Title: {job['title']}
Company: {job['company']}
Location: {job['location']}
Experience: {job['experience']}
Skills: {job['skills']}

Description:
{job['description']}

Similarity Score: {job['similarity_score']}
"""

    prompt += """

Choose the BEST 5 jobs.

For EACH selected job provide:

- Match Score (0-100)
- Fit Level (High / Medium / Low)
- Why it matches
- Matching Skills
- Missing Skills
- Career Advice


Return ONLY valid JSON in this format:

[
 {
   "title":"",
   "company":"",
   "match_score":95,
   "fit_level":"High",
   "why_match":"",
   "matching_skills":[],
   "missing_skills":[],
   "career_advice":""
Do not include markdown, code fences, explanations, or any text outside the JSON array.
 }
]

"""

    return prompt

prompt = create_prompt(resume_text, top_jobs)

In [19]:
from groq import Groq
client = Groq(api_key="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")

response = client.chat.completions.create(

    model="llama-3.3-70b-versatile",

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],

    temperature=0.2
)

result = response.choices[0].message.content

print(result)

[
  {
    "title": "Environment Officer",
    "company": "Balaji Amines",
    "match_score": 90,
    "fit_level": "High",
    "why_match": "The candidate's background in Environmental Science and experience in environmental consulting make them a strong fit for this role.",
    "matching_skills": ["Environmental science", "Environmental impact assessment", "Water treatment & Management"],
    "missing_skills": ["Chemical engineering", "ISO 14001"],
    "career_advice": "To be successful in this role, the candidate should focus on developing their knowledge of chemical engineering principles and ISO 14001 standards."
  },
  {
    "title": "Project Coordinator - Water Supply",
    "company": "Mar-haba Hr Solutions",
    "match_score": 88,
    "fit_level": "High",
    "why_match": "The candidate's experience in water supply projects and environmental consulting make them a strong fit for this role.",
    "matching_skills": ["Water Supply", "Project Coordination", "Water treatment & Manage

In [20]:
import json
import re

def safe_json_parse(response_text):
    try:
        return json.loads(response_text)

    except json.JSONDecodeError:

        cleaned = re.sub(r"```json|```", "", response_text).strip()

        try:
            return json.loads(cleaned)

        except json.JSONDecodeError:

            match = re.search(r"\[.*\]", cleaned, re.DOTALL)

            if match:
                return json.loads(match.group())

            match = re.search(r"\{.*\}", cleaned, re.DOTALL)

            if match:
                return json.loads(match.group())

            raise ValueError("Could not extract valid JSON.")

final_jobs = safe_json_parse(result)

In [21]:
for job in final_jobs:
    print("=" * 80)
    print("Title:", job["title"])
    print("Company:", job["company"])
    print("Match Score:", job["match_score"])
    print("Fit Level:", job["fit_level"])
    print("Why Match:", job["why_match"])
    print("Matching Skills:", job["matching_skills"])
    print("Missing Skills:", job["missing_skills"])
    print("Career Advice:", job["career_advice"])

Title: Environment Officer
Company: Balaji Amines
Match Score: 90
Fit Level: High
Why Match: The candidate's background in Environmental Science and experience in environmental consulting make them a strong fit for this role.
Matching Skills: ['Environmental science', 'Environmental impact assessment', 'Water treatment & Management']
Missing Skills: ['Chemical engineering', 'ISO 14001']
Career Advice: To be successful in this role, the candidate should focus on developing their knowledge of chemical engineering principles and ISO 14001 standards.
Title: Project Coordinator - Water Supply
Company: Mar-haba Hr Solutions
Match Score: 88
Fit Level: High
Why Match: The candidate's experience in water supply projects and environmental consulting make them a strong fit for this role.
Matching Skills: ['Water Supply', 'Project Coordination', 'Water treatment & Management']
Missing Skills: ['Qa/Qc', 'Irrigation Systems']
Career Advice: To be successful in this role, the candidate should focus o

In [46]:
!pip install groq